<a href="https://colab.research.google.com/github/Stronghold1388/Agent_experiments/blob/main/Personal_chef(Lanfchain_practice).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install -U langchain langchain-openai huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.9/132.9 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.8/119.8 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 693.4/693.4 kB 32.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 554.9/554.9 kB 35.3 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.4.3
    Uninstalling langchain-core-1.4.3:
      Successfully uninstalled langchain-core-1.4.3
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.18.0
    Uninstalling huggingface_hub-1.18.0:
      Successfully uninstalled huggingface_hub-1.18.0
  Attempting uninstall: langchain
    Found existing installation: langchain 1.3.6
    Uninstalling langchain-1.3.6:
      Successfully uninstalled langchain-1.3.6


In [26]:
#model_invocation
from huggingface_hub.inference._providers import openai
from langchain_openai.chat_models import ChatOpenAI
from huggingface_hub import InferenceClient
from google.colab import userdata
import os

client=InferenceClient(model='openai/gpt-oss-120b')
model=ChatOpenAI(
    openai_api_key=userdata.get('HF_TOKEN'),
    model_name='openai/gpt-oss-120b',
    base_url="https://router.huggingface.co/v1",
    max_tokens=300,
    temperature=0.7,
    streaming=True
)

In [22]:
!pip install tavily-python
from langchain.tools import tool
from tavily import TavilyClient
from typing import Dict,Any
from google.colab import userdata
import os

os.environ['TAVILY_API_KEY']=userdata.get('TAVILY_API_KEY')
tavily_client=TavilyClient()

#tools
@tool ('search_web',description='search the web for information')
def ws_tool(query,str):
  return tavily_client.search(query,max_results=3)

In [27]:

from langchain.agents import create_agent
from langchain.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

system_prompt="""You are a professional chef's assistant. You can suggest and improve recipe according to -occasion,deitary restrictions, demands, taste.
               You are able to use the given tools to reach best possible recipe for the user.
               you will always provide recipe in bullet points.
               you must suggest drinks that goes well with the suggested food.
               Always follow this output format=
               Recipe: (example-  Chicken curry
               -500g fresh chiken
               -ginger
               -garlic
               -100g onion ..etc)
               Drinks: (only name)
               Always follow this process-Think>Action>Observation(this process can repeat n times, till a satisfactory result is not achieved)"""

chef_agent=create_agent(model=model,
                        tools=[ws_tool],
                        system_prompt=system_prompt,
                        checkpointer=InMemorySaver() )

question=HumanMessage(content="Tommorow is my wife's bithday , she likes chicken dishes with rice.Can you suggest me some recipe for her?")
config={'configurable':{'thread_id':'1'}}

result=chef_agent.invoke(
                         {"messages":[question]},
                          config=config
                         )
print(result['messages'][-1].content)



**Recipe: Chicken Biryani (celebratory birthday dish)**
- 1 kg chicken thighs, cut into bite‑size pieces  
- 2 cups basmati rice, rinsed and soaked 30 min  
- 2 large onions, thinly sliced  
- 3 cloves garlic, minced  
- 1 in‑ch piece ginger, grated  
- 2 tomatoes, diced  
- 1 cup plain yogurt  
- 1 ½ cups chicken broth (low‑sodium)  
- ¼ cup vegetable oil or ghee  
- 2 tbsp biryani masala (store‑bought or homemade)  
- 1 tsp turmeric powder  
- 1 tsp cumin seeds  
- 1 tsp coriander seeds, lightly crushed  
- 1 tsp garam masala  
- ½ tsp saffron threads soaked in 2 tbsp warm milk  
- ¼ cup


In [34]:
question=HumanMessage(content='can you suggest someting light.')
config= {'configurable':{'thread_id':'2'}}

result=chef_agent.invoke(
                         {"messages":[question]},
                          config=config
                         )
print(result['messages'][-1].content)


**Recipe:** Zucchini Noodle Salad with Lemon‑Herb Vinaigrette  
- 3 large zucchini, spiralized into noodles  



In [35]:
question=HumanMessage(content='what about dessert.')
config= {'configurable':{'thread_id':'3'}}
result=chef_agent.invoke(
                         {"messages":[question]},
                          config=config
                         )
print(result['messages'][-1].content)

**Recipe:** Lemon‑Honey Chia Pudding with Fresh Berries  
- 3 tbsp chia seeds  
- 1 cup unsweetened almond milk (or any milk of choice)  
- 2 tbsp fresh lemon juice  
- 1 tbsp honey (or maple syrup for vegan option)  
- ½ tsp vanilla extract  
- Pinch of sea‑salt  
- ½ cup mixed fresh berries (strawberries, blueberries, raspberries)  
- Optional garnish: mint leaves and a light dusting of lemon zest  

**Drinks:** Sparkling Lemon‑Ginger Water


In [36]:
question=HumanMessage(content="don't you think with cake and all it would be toomuch sweet")

result=chef_agent.invoke(
                         {"messages":[question]},
                          config=config
                         )
print(result['messages'][-1].content)
#

**Recipe:** Lemon‑Ricotta Mousse (light, low‑sweetness)  
- 1 cup ricotta cheese (full‑fat works best for texture)  
- ½ cup plain Greek yogurt (for a tangy lift)  
- 2 tbsp fresh lemon juice  
- Zest of 1 lemon (optional, for extra aroma)  
- 1‑2 tbsp honey or maple syrup (adjust to taste; you can omit for virtually sugar‑free)  
- ½ tsp vanilla
